In [1]:
import os
import random
import shutil
from pathlib import Path

In [2]:
DATASET_DIR = r"data"  

IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")

print("Images folder exists:", os.path.exists(IMAGES_DIR))
print("Labels folder exists:", os.path.exists(LABELS_DIR))

Images folder exists: True
Labels folder exists: True


In [3]:
IMG_EXTS = [".jpg", ".jpeg", ".png"]

image_files = []
for f in os.listdir(IMAGES_DIR):
    if Path(f).suffix.lower() in IMG_EXTS:
        image_files.append(f)

print("Total images found:", len(image_files))
print("First 5 images:", image_files[:5])

Total images found: 2009
First 5 images: ['20250216_164325.jpg', '20250216_164521.jpg', '20250216_164541.jpg', '20250219_164649.jpg', '20250219_164714.jpg']


In [4]:
missing_labels = []

for img_name in image_files:
    txt_name = Path(img_name).stem + ".txt"
    txt_path = os.path.join(LABELS_DIR, txt_name)
    if not os.path.exists(txt_path):
        missing_labels.append(img_name)

print("Images without labels:", len(missing_labels))
print("Sample missing:", missing_labels[:10])

Images without labels: 0
Sample missing: []


In [5]:
matched_images = []

for img_name in image_files:
    txt_name = Path(img_name).stem + ".txt"
    txt_path = os.path.join(LABELS_DIR, txt_name)
    if os.path.exists(txt_path):
        matched_images.append(img_name)

print("Matched image-label pairs:", len(matched_images))

Matched image-label pairs: 2009


In [6]:
train_img_dir = os.path.join(DATASET_DIR, "images", "train")
val_img_dir   = os.path.join(DATASET_DIR, "images", "val")

train_lbl_dir = os.path.join(DATASET_DIR, "labels", "train")
val_lbl_dir   = os.path.join(DATASET_DIR, "labels", "val")

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(train_lbl_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

print("Folders created successfully")

Folders created successfully


In [7]:
random.seed(42)
random.shuffle(matched_images)

split_idx = int(0.8 * len(matched_images))

train_files = matched_images[:split_idx]
val_files   = matched_images[split_idx:]

print("Train files:", len(train_files))
print("Val files:", len(val_files))

Train files: 1607
Val files: 402


In [8]:
for img_name in train_files:
    src_img = os.path.join(IMAGES_DIR, img_name)
    src_lbl = os.path.join(LABELS_DIR, Path(img_name).stem + ".txt")

    dst_img = os.path.join(train_img_dir, img_name)
    dst_lbl = os.path.join(train_lbl_dir, Path(img_name).stem + ".txt")

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)

print("Train files copied")

Train files copied


In [9]:
# Copy val files
for img_name in val_files:
    src_img = os.path.join(IMAGES_DIR, img_name)
    src_lbl = os.path.join(LABELS_DIR, Path(img_name).stem + ".txt")

    dst_img = os.path.join(val_img_dir, img_name)
    dst_lbl = os.path.join(val_lbl_dir, Path(img_name).stem + ".txt")

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)

print("Validation files copied")

Validation files copied


In [10]:
print("Train images:", len(os.listdir(train_img_dir)))
print("Train labels:", len(os.listdir(train_lbl_dir)))
print("Val images:", len(os.listdir(val_img_dir)))
print("Val labels:", len(os.listdir(val_lbl_dir)))

Train images: 1607
Train labels: 1607
Val images: 402
Val labels: 402


In [11]:
# dataset.yaml create
yaml_content = """
path: 'D:/Intern Project/Final Project/data'
train: 'images/train'
val: 'images/val'

names:
  0: pothole
  1: crack
  2: manhole
"""

with open("dataset.yaml", "w") as f:
    f.write(yaml_content.strip())

print("dataset.yaml created")


dataset.yaml created


In [12]:
from ultralytics import YOLO 

In [13]:
# 1. Load the model (start fresh from pre-trained to avoid bias from bad training)
model = YOLO('yolov8n.pt')

In [14]:
results = model.train(data='dataset.yaml',    
    epochs=50,imgsz=640,batch=8,patience=5,project='road_damage_output',name='pothole_detection_fast',device='cpu')

New https://pypi.org/project/ultralytics/8.4.23 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.22  Python-3.12.1 torch-2.9.1+cpu CPU (11th Gen Intel Core i3-1115G4 @ 3.00GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pothole_detection_fa

In [19]:
metrics = model.val()

print("\n--- PERFORMANCE METRICS ---")
print(f"Overall Precision: {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Overall Recall:    {metrics.results_dict['metrics/recall(B)']:.4f}")
print(f"mAP50:             {metrics.results_dict['metrics/mAP50(B)']:.4f}")


Ultralytics 8.4.22  Python-3.12.1 torch-2.9.1+cpu CPU (11th Gen Intel Core i3-1115G4 @ 3.00GHz)
val: Fast image access  (ping: 0.00.0 ms, read: 143.759.6 MB/s, size: 98.8 KB)
val: Scanning D:\Intern Project\Final Project\data\labels\val.cache... 402 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 402/402  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 1.8s/it 47.1s1.9ss
                   all        402       1008      0.554      0.426      0.456        0.2
               pothole        162        263      0.518      0.357      0.424      0.175
                 crack        291        558      0.438      0.305      0.293      0.117
               manhole        140        187      0.704      0.615      0.651      0.308
Speed: 1.5ms preprocess, 105.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\Intern Project\Final Project\runs\detect\val5

--- PERFORMANCE METRICS ---
Overall Precisio

In [18]:
# Per-class breakdown
print("\n--- PER-CLASS BREAKDOWN ---")
names = model.names
for i, (p, r) in enumerate(zip(metrics.box.p, metrics.box.r)):
    print(f"Class: {names[i]:<10} | Precision: {p:.4f} | Recall: {r:.4f}")


--- PER-CLASS BREAKDOWN ---
Class: pothole    | Precision: 0.5176 | Recall: 0.3574
Class: crack      | Precision: 0.4385 | Recall: 0.3047
Class: manhole    | Precision: 0.7045 | Recall: 0.6150


In [23]:
import pandas as pd
import os
from glob import glob

# Automatically find the latest results file
all_runs = glob(os.path.join("runs", "detect", "road_damage_output", "*"))
if all_runs:
    latest_run = max(all_runs, key=os.path.getmtime)
    results_csv = os.path.join(latest_run, "results.csv")
    
    if os.path.exists(results_csv):
        # Read the training history
        df = pd.read_csv(results_csv)
        # Clean up column names (strip whitespace)
        df.columns = df.columns.str.strip()
        
        # Get the final epoch results
        final_results = df.iloc[-1]
        
        print(f"--- Final Evaluation Results (Epoch {int(final_results['epoch'])}) ---")
        print(f"Precision: {final_results['metrics/precision(B)']:.4f} ({final_results['metrics/precision(B)']*100:.2f}%)")
        print(f"Recall:    {final_results['metrics/recall(B)']:.4f} ({final_results['metrics/recall(B)']*100:.2f}%)")
        print(f"mAP50:     {final_results['metrics/mAP50(B)']:.4f}")
        print(f"mAP50-95:  {final_results['metrics/mAP50-95(B)']:.4f}")
    else:
        print("results.csv not found.")


--- Final Evaluation Results (Epoch 44) ---
Precision: 0.5554 (55.54%)
Recall:    0.4251 (42.51%)
mAP50:     0.4607
mAP50-95:  0.1986


In [15]:
# Create a path for sample validation images
val_images_path = os.path.join("data", "images", "val")

# Run prediction on the first 5 images in the validation set
results = model.predict(source=val_images_path, save=True, conf=0.25, imgsz=416, max_det=10)

print(f"Predictions completed. Results saved to: {results[0].save_dir}")



image 1/402 d:\Intern Project\Final Project\data\images\val\20250216_164521.jpg: 256x416 1 pothole, 1 crack, 474.7ms
image 2/402 d:\Intern Project\Final Project\data\images\val\20250219_164714.jpg: 256x416 1 pothole, 292.8ms
image 3/402 d:\Intern Project\Final Project\data\images\val\20250219_164746.jpg: 256x416 1 manhole, 144.6ms
image 4/402 d:\Intern Project\Final Project\data\images\val\20250219_164838.jpg: 256x416 1 pothole, 1 crack, 139.7ms
image 5/402 d:\Intern Project\Final Project\data\images\val\20250219_164848.jpg: 256x416 (no detections), 103.3ms
image 6/402 d:\Intern Project\Final Project\data\images\val\20250219_165056.jpg: 256x416 4 potholes, 99.6ms
image 7/402 d:\Intern Project\Final Project\data\images\val\20250219_165351.jpg: 256x416 2 potholes, 174.6ms
image 8/402 d:\Intern Project\Final Project\data\images\val\20250219_165753.jpg: 256x416 (no detections), 164.3ms
image 9/402 d:\Intern Project\Final Project\data\images\val\20250219_170258.jpg: 256x416 2 potholes, 2 c

In [25]:
best_model_path = r"runs/detect/road_damage_output/pothole_detection_fast4/weights/best.pt"

if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    # Run validation
    metrics = model.val()
    print("Validation Map50-95:", metrics.box.map) 
else:
    print(f"Could not find model at {best_model_path}. Please check the path.")


Ultralytics 8.4.22  Python-3.12.1 torch-2.9.1+cpu CPU (11th Gen Intel Core i3-1115G4 @ 3.00GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.2 ms, read: 82.553.2 MB/s, size: 100.6 KB)
val: Scanning D:\Intern Project\Final Project\data\labels\val.cache... 402 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 402/402 65.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 2.1s/it 54.8s2.0ss
                   all        402       1008      0.554      0.426      0.456        0.2
               pothole        162        263      0.518      0.357      0.424      0.175
                 crack        291        558      0.438      0.305      0.293      0.117
               manhole        140        187      0.704      0.615      0.651      0.308
Speed: 3.3ms preprocess, 120.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to D:\In